# Assignment 5 — Naive Bayes Classifiers

**Dataset:** Spambase (UCI) — classify emails as **spam (1)** or **not spam (0)**.

**Goal:** Compare three Naive Bayes variants:
- **GaussianNB** — for continuous features assumed to follow a Normal distribution.
- **MultinomialNB** — for discrete counts, e.g. word frequencies.
- **BernoulliNB** — for binary / boolean features.



In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, Binarizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, roc_auc_score)

# Ensure reproducibility
RANDOM_STATE = 42


## 1. Load the Spambase dataset

The dataset has 57 numeric features and a binary label in the last column (`1 = spam`, `0 = not spam`).


In [ ]:
# Load data
data_path = 'spambase.data'
df = pd.read_csv(data_path, header=None)

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print('Shape:', df.shape)
print('Class distribution:')
print(y.value_counts())


## 2. Train-Test Split and Preprocessing

- **GaussianNB** can work with raw continuous values, but we standardize for stability.
- **MultinomialNB** requires non-negative features. We use `MinMaxScaler` to scale values to [0, 1].
- **BernoulliNB** works best on binary features. We binarize the MinMax-scaled data with a threshold of 0.5.


In [ ]:
# Split into train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

# Preprocess for different NB variants

# GaussianNB: use standard-scaled continuous values
from sklearn.preprocessing import StandardScaler
scaler_g = StandardScaler()
X_train_g = scaler_g.fit_transform(X_train)
X_test_g = scaler_g.transform(X_test)

# MultinomialNB: requires non-negative values
scaler_m = MinMaxScaler()
X_train_m = scaler_m.fit_transform(X_train)
X_test_m = scaler_m.transform(X_test)

# BernoulliNB: binary features
binarizer = Binarizer(threshold=0.5)
X_train_b = binarizer.fit_transform(X_train_m)
X_test_b = binarizer.transform(X_test_m)


## 3. Train and Evaluate the three Naive Bayes models

In [ ]:
models = {
    'GaussianNB': (GaussianNB(), X_train_g, X_test_g),
    'MultinomialNB': (MultinomialNB(), X_train_m, X_test_m),
    'BernoulliNB': (BernoulliNB(), X_train_b, X_test_b),
}

results = []
trained_models = {}

for name, (model, X_tr, X_te) in models.items():
    t0 = time.time()
    model.fit(X_tr, y_train)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = model.predict(X_te)
    pred_time = time.time() - t0

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # ROC-AUC
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_te)[:, 1]
    else:
        y_prob = model.predict(X_te)
    auc = roc_auc_score(y_test, y_prob)

    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-score': round(f1, 4),
        'AUC': round(auc, 4),
        'Train time (s)': round(train_time, 4),
        'Predict time (s)': round(pred_time, 4)
    })

    trained_models[name] = {'model': model, 'y_pred': y_pred, 'y_prob': y_prob}

results_df = pd.DataFrame(results)
results_df


In [ ]:
print('Performance comparison:')
print(results_df.to_string(index=False))


## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ['GaussianNB', 'MultinomialNB', 'BernoulliNB']):
    y_pred = trained_models[name]['y_pred']
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['not spam', 'spam'],
                yticklabels=['not spam', 'spam'])
    ax.set_title(f'{name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()


## 5. ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))
for name in ['GaussianNB', 'MultinomialNB', 'BernoulliNB']:
    y_prob = trained_models[name]['y_prob']
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Naive Bayes variants')
plt.legend()
plt.tight_layout()
plt.show()


## 6. When to use each Naive Bayes variant

### GaussianNB
- **Use when:** features are continuous and approximately normally distributed.
- **Example:** sensor readings, real-valued medical measurements, age, income.
- In this notebook we standardize the Spambase word-frequency features before passing them to `GaussianNB`.

### MultinomialNB
- **Use when:** features are discrete counts or frequencies.
- **Example:** word counts, term frequencies in text classification.
- It requires **non-negative** input, so we scale the data with `MinMaxScaler` to [0, 1].

### BernoulliNB
- **Use when:** features are binary / boolean.
- **Example:** presence (1) or absence (0) of a word in a document.
- We first `MinMaxScaler` the data and then `Binarizer` it at threshold 0.5 to get 0/1 features.

## 7. Conclusion
- The three NB variants make different assumptions about feature distributions.
- `MultinomialNB` and `BernoulliNB` are common in text/spam classification.
- `GaussianNB` can still work here because the Spambase features are continuous word-frequencies.
- Compare the accuracy, precision, recall, F1 and AUC to decide the best model for this task.
